In [2]:
import pandas as pd

df = pd.read_csv("../data/processed_deliveries_normalized.csv")
df.head()


,match_id,season,start_date,venue,innings,ball,batting_team,bowling_team,striker,non_striker,...,byes,legbyes,penalty,wicket_type,player_dismissed,other_wicket_type,other_player_dismissed,total_runs,wicket,cumulative_runs
0,335982,2008,18-04-2008,M Chinnaswamy Stadium,0.0,0.1,Kolkata Knight Riders,Royal Challengers Bangalore,SC Ganguly,BB McCullum,...,0.0,1.0,0.0,Unknown,Unknown,0.0,0.0,0.142857,0.0,0.0
1,335982,2008,18-04-2008,M Chinnaswamy Stadium,0.0,0.2,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,...,0.0,0.0,0.0,Unknown,Unknown,0.0,0.0,0.000000,0.0,0.0
2,335982,2008,18-04-2008,M Chinnaswamy Stadium,0.0,0.3,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,...,0.0,0.0,0.0,Unknown,Unknown,0.0,0.0,0.142857,0.0,0.0
3,335982,2008,18-04-2008,M Chinnaswamy Stadium,0.0,0.4,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,...,0.0,0.0,0.0,Unknown,Unknown,0.0,0.0,0.000000,0.0,0.0
4,335982,2008,18-04-2008,M Chinnaswamy Stadium,0.0,0.5,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,...,0.0,0.0,0.0,Unknown,Unknown,0.0,0.0,0.000000,0.0,0.0


In [3]:
df.shape
df.columns


Index(['match_id', 'season', 'start_date', 'venue', 'innings', 'ball',
       'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler',
       'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes',
       'penalty', 'wicket_type', 'player_dismissed', 'other_wicket_type',
       'other_player_dismissed', 'total_runs', 'wicket', 'cumulative_runs'],
      dtype='object')

In [4]:
batsman_match = (
    df.groupby(
        ["match_id", "striker", "batting_team", "bowling_team", "venue"]
    )
    .agg(
        runs=("runs_off_bat", "sum"),
        balls=("runs_off_bat", "count")
    )
    .reset_index()
)

batsman_match["strike_rate"] = (batsman_match["runs"] / batsman_match["balls"]) * 100
batsman_match.head()


,match_id,striker,batting_team,bowling_team,venue,runs,balls,strike_rate
0,335982,AA Noffke,Royal Challengers Bangalore,Kolkata Knight Riders,M Chinnaswamy Stadium,1.500000,12,12.500000
1,335982,B Akhil,Royal Challengers Bangalore,Kolkata Knight Riders,M Chinnaswamy Stadium,0.000000,2,0.000000
2,335982,BB McCullum,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,26.333333,77,34.199134
3,335982,CL White,Royal Challengers Bangalore,Kolkata Knight Riders,M Chinnaswamy Stadium,1.000000,10,10.000000
4,335982,DJ Hussey,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,2.000000,12,16.666667


In [5]:
batsman_match = batsman_match.sort_values(
    ["striker", "match_id"]
)


In [6]:
batsman_match["runs_next_match"] = (
    batsman_match.groupby("striker")["runs"].shift(-1)
)


In [7]:
batsman_match = batsman_match.dropna(subset=["runs_next_match"])


In [8]:
batsman_match["avg_runs_last_5"] = (
    batsman_match.groupby("striker")["runs"]
    .rolling(5, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)


In [9]:
batsman_match["venue_avg_runs"] = (
    batsman_match.groupby(["striker", "venue"])["runs"]
    .transform("mean")
)


In [10]:
batsman_match["opponent_avg_runs"] = (
    batsman_match.groupby(["striker", "bowling_team"])["runs"]
    .transform("mean")
)


In [11]:
batsman_match["career_avg_runs"] = (
    batsman_match.groupby("striker")["runs"]
    .transform("mean")
)

batsman_match["matches_played"] = (
    batsman_match.groupby("striker").cumcount() + 1
)


In [12]:
features = [
    "avg_runs_last_5",
    "venue_avg_runs",
    "opponent_avg_runs",
    "career_avg_runs",
    "matches_played",
    "strike_rate"
]

target = "runs_next_match"

final_df = batsman_match[features + [target]]
final_df.head()


,avg_runs_last_5,venue_avg_runs,opponent_avg_runs,career_avg_runs,matches_played,strike_rate,runs_next_match
4299,1.666667,1.666667,2.250000,2.106061,1,16.666667,0.500000
4390,1.083333,3.250000,2.500000,2.106061,2,16.666667,1.333333
4496,1.166667,1.516667,2.055556,2.106061,3,16.666667,1.666667
4699,1.291667,1.516667,2.055556,2.106061,4,41.666667,0.666667
4747,1.166667,1.516667,2.208333,2.106061,5,13.333333,1.166667


In [13]:
split_index = int(len(final_df) * 0.8)

X_train = final_df.iloc[:split_index][features]
X_test = final_df.iloc[split_index:][features]

y_train = final_df.iloc[:split_index][target]
y_test = final_df.iloc[split_index:][target]


In [14]:
final_df.to_csv("../data/final_batsman_features.csv", index=False)


In [17]:


import pandas as pd

# Load the processed dataset
df = pd.read_csv("../data/final_batsman_features.csv")

# Split features and target
target_col = 'runs_next_match'
X = df.drop(target_col, axis=1)
y = df[target_col]

# Optional: If you want a train-test split here
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)


X_train shape: (11810, 6)
X_test shape: (2953, 6)


In [18]:


from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib

# Step 1: Create a pipeline
preprocessor = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Step 2: Fit pipeline on training data
preprocessor.fit(X_train)

# Step 3: Transform train and test sets
X_train_scaled = preprocessor.transform(X_train)
X_test_scaled = preprocessor.transform(X_test)

# Step 4: Save the pipeline
joblib.dump(preprocessor, "../featurepipeline.pkl")
print("Feature pipeline saved as featurepipeline.pkl")


Feature pipeline saved as featurepipeline.pkl


In [22]:
# Run this later in the same notebook or a new notebook/script

# Load pipeline
preprocessor = joblib.load("../featurepipeline.pkl")

# Example: new match data (same columns as training features)
new_data = pd.read_csv("../data/new_match_features.csv")
new_data_scaled = preprocessor.transform(new_data)

print(new_data_scaled)


FileNotFoundError: [Errno 2] No such file or directory: '../data/new_match_features.csv'